# 01 — Explore the data

Purpose: look at real and fake face samples, check class balance, and check the
raw image sizes before the training pipeline resizes everything. See
[docs/03_data.md](../docs/03_data.md) for the dataset itself
(Kaggle `xhlulu/140k-real-and-fake-faces`: real faces from FFHQ, fake faces from
StyleGAN). Labels: `fake` = 1, `real` = 0.

This notebook only reads data; it never trains or writes anything. If the dataset is not on this machine yet, Step 2 lets you open an existing
folder/zip or upload a zip (notebook 05 can also download it from Kaggle).
Run the cells from top to bottom. A GPU is not needed just to explore the data.

## Step 1 — Find the project and data

**What this does:** finds the repository, loads the shared data-folder setting, creates the empty dataset folder skeleton if it is missing, and checks whether the `train`, `valid`, and `test` splits already contain images.

**Why:** using the config avoids hard-coding the dataset location. Adding `src/` to Python's import path lets this notebook import the project's own data helpers. The Colab fallback clones the public project into `/content` only if the notebook was opened without the repository files; on your own machine, with the notebook inside the project, nothing is cloned or created except the empty data folders. This cell does not download the dataset.

The dataset itself (about 4 GB, see [docs/03_data.md](../docs/03_data.md)) is never committed to git. The expected layout is:

```
<project root>/data/raw/real_vs_fake/real-vs-fake/
├── train/real/*.jpg   train/fake/*.jpg
├── valid/real/*.jpg   valid/fake/*.jpg
└── test/real/*.jpg    test/fake/*.jpg
```

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_NAME = "UOT_SCS_3546_041_Deep-Learning"
REPO_URL = "https://github.com/msondhi/UOT_SCS_3546_041_Deep-Learning.git"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}
SPLITS = ("train", "valid", "test")
CLASSES = ("real", "fake")

def find_project_root():
    return next(
        (path for path in (Path.cwd(), *Path.cwd().parents)
         if (path / "src" / "deepfake_kyc").is_dir()),
        None,
    )

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    try:
        import google.colab  # noqa: F401
    except ImportError as exc:
        raise FileNotFoundError(
            "Open this notebook from the project folder, or run it in Colab so it can clone the project."
        ) from exc
    PROJECT_ROOT = Path("/content") / REPO_NAME
    if not PROJECT_ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
    if not (PROJECT_ROOT / "src" / "deepfake_kyc").is_dir():
        raise FileNotFoundError(f"Project source folder not found after clone: {PROJECT_ROOT}")

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from deepfake_kyc.config import load_config

cfg = load_config(PROJECT_ROOT / "configs" / "base.yaml")
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
DATA_ROOT = PROJECT_ROOT / cfg["data"]["root"]
split_paths = {split: DATA_ROOT / split for split in SPLITS}

# Create the empty folder skeleton so you can see where the images belong.
for split_dir in split_paths.values():
    for class_name in CLASSES:
        (split_dir / class_name).mkdir(parents=True, exist_ok=True)

def count_images(folder):
    """Number of image files directly inside ``folder`` (placeholder files such as .gitkeep are ignored)."""
    return sum(1 for p in folder.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS) if folder.is_dir() else 0

def report_data_status():
    """Print how many images each split has and set DATA_AVAILABLE (every split needs real AND fake images)."""
    global DATA_AVAILABLE
    ready = True
    for split, path in split_paths.items():
        n_real, n_fake = count_images(path / "real"), count_images(path / "fake")
        ok = n_real > 0 and n_fake > 0
        ready = ready and ok
        print(f"{split:6} {'ready' if ok else 'EMPTY'}  real={n_real:,}  fake={n_fake:,}  {path}")
    DATA_AVAILABLE = ready
    print("Dataset ready." if ready else "Dataset not complete yet: add images with Step 2 below.")

print(f"Project root: {PROJECT_ROOT}")
report_data_status()

## Step 2 — Add the dataset (open an existing folder or zip, or upload a zip)

**What this does:** shows two controls. **Open path** takes a path you type (a folder that already contains `train`/`valid`/`test`, or the Kaggle `.zip`) and either uses that folder for this notebook or extracts the zip into `data/raw/`. **Upload zip** opens your browser's file chooser and extracts the chosen zip into `data/raw/`. Both finish by re-running the Step 1 check. Run the cell and the buttons appear below it.

**Why:** the dataset is too large for GitHub, so each machine (or each new Colab runtime) needs its own copy. The Kaggle zip already contains the `real_vs_fake/real-vs-fake/{train,valid,test}/{real,fake}` folders, so extracting it into `data/raw/` produces the expected layout automatically. Skip this step if Step 1 already printed `Dataset ready.`

**Size warning:** the browser upload button holds the whole file in memory, so use it only for small zips (for example a subset you made yourself). For the full 4 GB dataset, download the zip from Kaggle with your browser, place it anywhere on disk (or directly in `data/raw/`), and use **Open path** instead. On Colab, the Kaggle download cell in notebook 05 is the faster route, and files under `/content` disappear when the runtime resets.

In [ ]:
import os
import zipfile

from IPython.display import display


def find_split_root(start, max_depth=4):
    """Return the first folder at or below ``start`` that contains train/, valid/ and test/ subfolders."""
    start = Path(start)
    base_depth = len(start.parts)
    for current, dirs, _files in os.walk(start):
        if all(name in dirs for name in SPLITS):
            return Path(current)
        if len(Path(current).parts) - base_depth >= max_depth:
            dirs[:] = []
    return None

def use_folder(folder):
    """Point this notebook's split paths at ``folder`` (which must contain train/valid/test)."""
    global DATA_ROOT, split_paths
    found = find_split_root(folder)
    if found is None:
        print(f"No train/valid/test folders found under {folder}.")
        return
    DATA_ROOT = found
    split_paths = {split: DATA_ROOT / split for split in SPLITS}
    expected = PROJECT_ROOT / cfg["data"]["root"]
    if found.resolve() != expected.resolve():
        print(f"Using {found} for this notebook only.")
        print(f"To make the training scripts find it too, move it to {expected}")
        print("or set data.root in configs/base.yaml to that path.")
    report_data_status()

def extract_zip(zip_path):
    """Extract a dataset zip into data/raw/ and re-check the splits."""
    zip_path = Path(zip_path)
    if not zipfile.is_zipfile(zip_path):
        print(f"Not a valid zip file: {zip_path}")
        return
    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {zip_path.name} into {RAW_DATA_DIR} ... (the full dataset takes a few minutes)")
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(RAW_DATA_DIR)
    print("Extraction finished.")
    use_folder(RAW_DATA_DIR)

def add_dataset_from_path(text):
    """Dispatch a typed path: zip files are extracted, folders are used in place."""
    path = Path(text.strip().strip('"')).expanduser()
    if not text.strip():
        print("Type a path first.")
    elif path.is_file() and path.suffix.lower() == ".zip":
        extract_zip(path)
    elif path.is_dir():
        use_folder(path)
    else:
        print(f"Path not found or not a zip/folder: {path}")

try:
    import ipywidgets as widgets
except ImportError:
    widgets = None
    print("ipywidgets is not installed, so the buttons cannot be shown.")
    print('Install it with:  pip install ipywidgets   (or pip install -e ".[dev]")')
    print("Without the buttons you can still call:  add_dataset_from_path(r'C:/path/to/archive.zip or folder')")

if widgets is not None:
    path_box = widgets.Text(
        placeholder="e.g. C:/Downloads/archive.zip  or  /content/drive/MyDrive/real-vs-fake",
        description="Path:",
        layout=widgets.Layout(width="70%"),
    )
    open_button = widgets.Button(description="Open path", button_style="primary", icon="folder-open")
    upload_button = widgets.FileUpload(accept=".zip", multiple=False, description="Upload zip", icon="upload")
    log = widgets.Output()

    def on_open_clicked(_button):
        with log:
            log.clear_output()
            add_dataset_from_path(path_box.value)

    def on_upload_changed(change):
        uploaded = change["new"]
        if not uploaded:
            return
        # ipywidgets 8 gives a tuple of dicts; ipywidgets 7 (Colab) gives {filename: {"content": ...}}.
        if isinstance(uploaded, dict):
            name, item = next(iter(uploaded.items()))
            content = item["content"]
        else:
            item = uploaded[0]
            name, content = item["name"], item["content"]
        with log:
            log.clear_output()
            RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
            target = RAW_DATA_DIR / Path(name).name
            target.write_bytes(bytes(content))
            print(f"Saved upload to {target} ({target.stat().st_size / 1e6:.1f} MB)")
            extract_zip(target)
        upload_button.value = {} if isinstance(uploaded, dict) else ()

    open_button.on_click(on_open_clicked)
    upload_button.observe(on_upload_changed, names="value")

    display(widgets.HTML("<b>Add the dataset:</b> type a path to a folder or zip and click <i>Open path</i>, "
                         "or click <i>Upload zip</i> to choose a small zip from your computer."))
    display(widgets.HBox([path_box, open_button]), upload_button, log)

## Step 3 — Count real and fake images

**What this does:** counts each class in every available split and plots the counts.

**Why:** the count can reveal missing folders or class imbalance. The project helper maps `fake` to label `1` and `real` to `0`. Keep the test split for final evaluation; use training/validation data for development.

In [ ]:
if DATA_AVAILABLE:
    import matplotlib.pyplot as plt

    from deepfake_kyc.data import make_dataset

    counts_by_split = {}
    for split, path in split_paths.items():
        ds = make_dataset(path)
        if split == "train":
            train_ds = ds
        fake_count = sum(ds.target_transform(target) for _, target in ds.samples)
        counts_by_split[split] = {"real": len(ds) - fake_count, "fake": fake_count}
        counts = counts_by_split[split]
        print(f"{split:6} total={len(ds):,} | real (0)={counts['real']:,} | fake (1)={counts['fake']:,}")

    fig, ax = plt.subplots(figsize=(6, 3.5))
    splits = list(counts_by_split)
    x = range(len(splits))
    ax.bar([i - 0.2 for i in x], [counts_by_split[s]['real'] for s in splits], width=0.4, label="real (0)")
    ax.bar([i + 0.2 for i in x], [counts_by_split[s]['fake'] for s in splits], width=0.4, label="fake (1)")
    ax.set_xticks(list(x), splits)
    ax.set_ylabel("images")
    ax.set_title("Class balance per split")
    ax.legend()
    fig.tight_layout()
else:
    print(f"Dataset not found under {DATA_ROOT}. Add it with Step 2 (or see docs/03_data.md), then rerun from Step 1.")

## Step 4 — Check original image sizes

**What this does:** randomly checks up to 200 training images and counts their original `(width, height)` sizes.

**Why:** images may have different source sizes. The training pipeline resizes them to the configured square size (`224×224`) before the model sees them. Sampling is enough to spot common size patterns without opening every file.

In [ ]:
if DATA_AVAILABLE:
    import random
    from collections import Counter

    from PIL import Image

    rng = random.Random(0)
    sample_paths = [p for p, _ in rng.sample(train_ds.samples, min(200, len(train_ds.samples)))]
    sizes = Counter()
    for path in sample_paths:
        with Image.open(path) as image:
            sizes[image.size] += 1

    print(f"Sampled {len(sample_paths)} training images.")
    print("Image (width, height) -> count:")
    for size, count in sizes.most_common():
        print(f"  {size}: {count}")
    print(
        f"\nConfigured training resolution: {cfg['data']['image_size']}x{cfg['data']['image_size']} "
        "(images are resized to this before the model sees them)."
    )
else:
    print("Skipped: dataset not available.")


## Step 5 — View a few examples

**What this does:** displays up to four randomly selected training images from each class.

**Why:** this is a quick visual check for corrupt files, unexpected content, and obvious labeling problems. A handful of examples cannot tell us whether the model will work; visual differences can be subtle, and the sample should not replace quantitative evaluation.

## Sample grid

A quick look at a few real and fake faces side by side.


In [ ]:
if DATA_AVAILABLE:
    import random

    rng = random.Random(0)
    shown_per_class = 4
    real_paths = [p for p, target in train_ds.samples if train_ds.target_transform(target) == 0]
    fake_paths = [p for p, target in train_ds.samples if train_ds.target_transform(target) == 1]
    reals = rng.sample(real_paths, min(shown_per_class, len(real_paths)))
    fakes = rng.sample(fake_paths, min(shown_per_class, len(fake_paths)))
    chosen = [(p, "real") for p in reals] + [(p, "fake") for p in fakes]

    fig, axes = plt.subplots(1, len(chosen), figsize=(2.2 * len(chosen), 2.6), squeeze=False)
    for ax, (path, label) in zip(axes.ravel(), chosen):
        with Image.open(path) as image:
            ax.imshow(image.convert("RGB"))
        ax.set_title(label)
        ax.axis("off")
    fig.tight_layout()
else:
    print("Skipped: dataset not available.")


## Next

[02_degradations_gallery.ipynb](02_degradations_gallery.ipynb) shows what these
faces look like under the robustness degradations (blur, JPEG, downscale,
noise, screen capture).

## Quick notes to remember

- `train` teaches the model; `valid` helps choose the model; `test` is held back for the final score.
- `fake = 1` is the positive class and `real = 0` is the negative class.
- The size check samples at most 200 training images; its counts are not a full-dataset audit.
- The sample grid is only a sanity check. Do not infer detection accuracy from a few pictures.
- Record the actual class counts and common image sizes from your run in your report; this notebook does not assume the documented totals are present locally.
